<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/ReAct%26RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Engine 선택

In [1]:
LLM_PROVIDER = "openai" # claude 또는 gemini
KEYFROM_USERDATA = True  # 외부에서 입력하려면 False 로
TEMPERATURE = 0.2

필요한 모듈을 설치한다.

In [ ]:
pip install -qU requests beautifulsoup4 pandas langchain langchain-core langchain_openai langchain-anthropic langchain-google-genai

필요한 모듈 임포트

In [3]:
import re
import json
import time
import getpass
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag

from IPython.display import display
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

Open API 키를 입력 받는다.

In [4]:
if KEYFROM_USERDATA:
  from google.colab import userdata
  API_KEY = userdata.get('OPENAI').strip()
else:
  import getpass
  API_KEY = getpass.getpass('Enter API key: ')

LLM을 생성한다.

In [5]:
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

elif LLM_PROVIDER == "claude":
    from langchain_anthropic import ChatAnthropic

    llm = ChatAnthropic(
        model="claude-sonnet-4-6",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )


elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=TEMPERATURE,
        api_key=API_KEY,
      )

질문을 입력받는다. 질문이 없으면 디폴트 질문은 "aSSIST 주소 알려줘"

In [6]:
user_question = input("\n질문 EX) aSSIST 주소가 뭐야? ").strip()

if not user_question:
    user_question = "aSSIST 주소가 뭐야?"


질문 EX) aSSIST 주소가 뭐야? 


검색에 사용할 홈페이지 주소를 입력 받는다.

In [ ]:
homepage_url = input("\n검색할 홈페이지 메인 URL을 입력하세요: ").strip()

if not homepage_url:
    raise ValueError("홈페이지 URL이 필요합니다.")

if not homepage_url.startswith("http"):
    homepage_url = "https://" + homepage_url

print("\n[User Question]")
print(user_question)

print("\n[Homepage URL]")
print(homepage_url)

프롬프트를 구성한다.

In [8]:
baseline_prompt = ChatPromptTemplate.from_template("""
너는 신중한 assistant다.

사용자 질문:
{user_question}

외부 검색, 홈페이지 확인, 위키피디아 확인 없이 답해야 한다.
확실하지 않으면 모른다고 말하라.
추측하지 말라.

[출력 형식]
## Baseline Answer
- 답변:
- 불확실한 이유:
- 추가로 필요한 정보:
""")

RAG를 전혀 사용하지 않은 상태의 대답을 확인해 본다.

In [ ]:
baseline_chain = baseline_prompt | llm | StrOutputParser()

baseline_answer = baseline_chain.invoke({
    "user_question": user_question
})

print("\n" + "=" * 100)
print("[Baseline Agent Output]")
print("=" * 100)
print(baseline_answer)

Agent 1: Planner Agent <BR>
 질문을 보고 어떤 정보를 찾아야 하는지 계획하도록 지시한다.

In [10]:
planner_prompt = ChatPromptTemplate.from_template("""
너는 Homepage QA Planner Agent다.

사용자 질문을 해결하기 위해 홈페이지 내부 탐색 계획을 세워라.

사용자 질문:
{user_question}

검색 대상 홈페이지:
{homepage_url}

해야 할 일:
- 질문 유형을 분류한다.
- 사용자가 찾는 대상 또는 정보 항목을 정리한다.
- 홈페이지 내부 검색에 사용할 키워드를 만든다.
- 우선적으로 탐색할 URL/메뉴 키워드를 만든다.
- 최종 답변에서 확인해야 할 필드를 만든다.

질문 유형 예시:
- address: 주소, 위치, 오시는 길
- contact: 전화번호, 이메일, 문의처
- person: 특정 인물, 교수, 연구자
- faculty: 교수진, 교원 목록
- admission: 입학, 모집요강, 전형
- program: 과정, 학위, 교육 프로그램
- tuition: 등록금, 수업료
- research: 연구, 논문, 연구센터
- notice: 공지사항
- general: 기타 일반 질문

반드시 JSON만 출력하라.
마크다운 코드블록 금지.
JSON 밖 설명 금지.

{{
  "agent_name": "Planner Agent",
  "question_type": "address/contact/person/faculty/admission/program/tuition/research/notice/general 중 하나",
  "target_information": "사용자가 찾는 정보",
  "search_keywords": ["검색어1", "검색어2", "검색어3"],
  "priority_url_keywords": ["contact", "location", "about", "admission", "faculty 등"],
  "answer_fields": ["최종 답변에서 확인해야 할 필드1", "필드2"],
  "search_strategy": "홈페이지 메인에서 내부 링크를 수집한 뒤, 질문 유형과 키워드에 맞는 후보 페이지를 찾는다."
}}
""")

호출한 다음 답변을 확인해 본다.

In [ ]:
planner_chain = planner_prompt | llm | StrOutputParser()

planner_raw = planner_chain.invoke({
    "user_question": user_question,
    "homepage_url": homepage_url
})

print("\n" + "=" * 100)
print("[Agent 1 Output] Planner Agent")
print("=" * 100)
print(planner_raw)

LLM 프래너가 뽑아 놓은 키워드를 사용해 검색한다.<BR>
추가적으로 원질문의 키워드도 사용한다.

In [ ]:
planner_clean = planner_raw.strip()
planner_clean = planner_clean.replace("```json", "")
planner_clean = planner_clean.replace("```", "")
planner_clean = planner_clean.strip()

try:
    planner_result = json.loads(planner_clean)

except Exception as e:
    print("\nPlanner JSON 파싱 실패. fallback 키워드를 사용합니다.")
    print("에러:", e)

    planner_result = {
        "search_keywords": re.findall(r"[가-힣A-Za-z0-9]{2,}", user_question),
        "priority_url_keywords": []
    }

search_keywords = planner_result.get("search_keywords", [])
priority_url_keywords = planner_result.get("priority_url_keywords", [])


# 질문 자체에서 나온 단어도 검색어에 추가
question_terms = re.findall(r"[가-힣A-Za-z0-9]{2,}", user_question)

for term in question_terms:
    if term not in search_keywords:
        search_keywords.append(term)

# 자주 나오는 질문 유형에 대한 보조 키워드만 간단히 추가
if "주소" in user_question or "위치" in user_question or "어디" in user_question:
    search_keywords += ["주소", "위치", "오시는", "오시는길", "찾아오시는", "location", "address", "map"]

if "연락" in user_question or "전화" in user_question or "문의" in user_question:
    search_keywords += ["연락처", "전화", "문의", "이메일", "contact", "tel", "email"]

if "교수" in user_question or "교수진" in user_question:
    search_keywords += ["교수", "교수진", "faculty", "professor", "profile"]

if "입학" in user_question or "전형" in user_question or "모집" in user_question:
    search_keywords += ["입학", "전형", "모집", "admission", "apply"]

# 중복 제거
search_keywords = list(dict.fromkeys([str(k).strip() for k in search_keywords if str(k).strip()]))
priority_url_keywords = list(dict.fromkeys([str(k).strip() for k in priority_url_keywords if str(k).strip()]))

print("\n[Parsed Planner Result]")
print("search_keywords:", search_keywords)
print("priority_url_keywords:", priority_url_keywords)

홈페이지를 따라 링크를 크롤링한다.

In [ ]:
print("\n" + "=" * 100)
print("[Tool Execution] Crawl Homepage Tool")
print("=" * 100)

MAX_PAGES = 5    # 최대 5개 페이지 검색
REQUEST_DELAY = 0.2
TIMEOUT = 10

# 이 예제는 그림 파일은 검색에서 제외해서 속도를 올린다.
EXCLUDE_EXTENSIONS = (
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg",
    ".zip", ".hwp", ".hwpx", ".doc", ".docx", ".ppt", ".pptx",
    ".xls", ".xlsx", ".mp4", ".mp3"
)

# 이 헤더를 같이 보내지 않으면 서버가 BOT으로 인식해 차단할 수 있다.
headers = {
    "User-Agent": "Mozilla/5.0 Simple-Homepage-QA-Classroom/1.0"
}

# 헤더를 최우선으로 검색한다.
FOOTER_SELECTORS = [
    "footer",
    "#footer",
    ".footer",
    "[id*='footer']",
    "[class*='footer']",
    "[id*='Footer']",
    "[class*='Footer']",
]

FOOTER_LINK_TERMS = [
    "footer", "contact", "location", "map", "directions", "address",
    "오시는", "오시는길", "찾아오시는", "위치", "주소", "연락처", "문의"
]

def normalize_url(base_url, link):
    joined = urljoin(base_url, link)
    joined, _ = urldefrag(joined)
    return joined.rstrip("/")

def site_key(url):
    return urlparse(url).netloc.lower().replace("www.", "")

def is_same_site(url, root_key):
    return site_key(url) == root_key

def is_crawlable_url(url):
    lower = url.lower()

    if not lower.startswith("http"):
        return False

    if "mailto:" in lower or "javascript:" in lower:
        return False

    if any(lower.endswith(ext) for ext in EXCLUDE_EXTENSIONS):
        return False

    return True

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def extract_footer_info(soup, base_url):
    footer_texts = []
    footer_links = []

    footer_nodes = []

    for selector in FOOTER_SELECTORS:
        footer_nodes.extend(soup.select(selector))

    # 중복 footer node 제거
    seen_node_ids = set()
    unique_footer_nodes = []

    for node in footer_nodes:
        node_id = id(node)

        if node_id not in seen_node_ids:
            seen_node_ids.add(node_id)
            unique_footer_nodes.append(node)

    for footer in unique_footer_nodes:
        footer_text = footer.get_text(" ", strip=True)
        footer_text = clean_text(footer_text)

        if footer_text:
            footer_texts.append(footer_text)

        for a in footer.find_all("a", href=True):
            href = a.get("href")
            anchor = a.get_text(" ", strip=True)
            next_url = normalize_url(base_url, href)

            footer_links.append({
                "url": next_url,
                "anchor": anchor,
                "from_footer": True
            })

    footer_text = "\n".join(footer_texts)
    footer_text = clean_text(footer_text)

    return footer_text, footer_links


def fetch_page(url):
    try:
        response = requests.get(url, headers=headers, timeout=TIMEOUT)

        if response.status_code != 200:
            return None, []

        content_type = response.headers.get("Content-Type", "")

        if "text/html" not in content_type and "application/xhtml+xml" not in content_type:
            return None, []

        if not response.encoding:
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        title = soup.title.get_text(" ", strip=True) if soup.title else ""

        # 1) footer 먼저 별도 추출
        footer_text, footer_links = extract_footer_info(soup, url)

        # 2) 전체 텍스트 추출
        text = soup.get_text(" ", strip=True)
        text = clean_text(text)

        # 3) footer 태그가 명시적으로 안 잡히는 사이트 대비:
        #    전체 텍스트의 마지막 부분을 fallback_footer_text로 둔다.
        fallback_footer_text = text[-2500:] if len(text) > 2500 else text

        if not footer_text:
            footer_text = fallback_footer_text

        # 4) 일반 링크 추출
        links = []

        # footer 링크를 먼저 넣음
        links.extend(footer_links)

        for a in soup.find_all("a", href=True):
            href = a.get("href")
            anchor = a.get_text(" ", strip=True)
            next_url = normalize_url(url, href)

            links.append({
                "url": next_url,
                "anchor": anchor,
                "from_footer": False
            })

        page_data = {
            "url": url,
            "title": title,
            "text": text,
            "footer_text": footer_text,
            "text_length": len(text),
            "footer_length": len(footer_text)
        }

        return page_data, links

    except Exception:
        return None, []


def link_priority(url, anchor, terms):
    combined = (url + " " + anchor).lower()
    score = 0

    for term in terms:
        term = str(term).lower()

        if term and term in combined:
            score += 10

    return score


root_key = site_key(homepage_url)
start_url = normalize_url(homepage_url, "")

priority_terms = list(dict.fromkeys(priority_url_keywords + search_keywords))

visited = set()
queued = set([start_url])
frontier = [(100, start_url)]
pages = []

while frontier and len(pages) < MAX_PAGES:
    frontier = sorted(frontier, key=lambda x: x[0], reverse=True)
    _, current_url = frontier.pop(0)

    if current_url in visited:
        continue

    visited.add(current_url)

    if not is_same_site(current_url, root_key):
        continue

    if not is_crawlable_url(current_url):
        continue

    print(f"크롤링 중 [{len(pages) + 1}/{MAX_PAGES}]: {current_url}")

    page_data, links = fetch_page(current_url)

    if page_data is not None and page_data["text_length"] > 0:
        pages.append(page_data)

    for link_info in links:
        next_url = link_info["url"]
        anchor = link_info["anchor"]

        if next_url in visited or next_url in queued:
            continue

        if not is_same_site(next_url, root_key):
            continue

        if not is_crawlable_url(next_url):
            continue

        score = link_priority(next_url, anchor, priority_terms)
        frontier.append((score, next_url))
        queued.add(next_url)

    time.sleep(REQUEST_DELAY)

pages_df = pd.DataFrame(pages)

if len(pages_df) == 0:
    raise RuntimeError("홈페이지에서 페이지를 수집하지 못했습니다. URL 또는 접근 제한을 확인하세요.")

print("\n수집된 페이지 수:", len(pages_df))
display(pages_df[["title", "url", "text_length"]].head(40))


수집한 페이지에서 질문 키워드와 가장 많이 맞는 페이지를 찾는다.

In [ ]:
print("\n" + "=" * 100)
print("[Tool Execution] Simple Page Search Tool")
print("=" * 100)

def make_snippet(text, keywords, window=400):
    text_lower = text.lower()

    for kw in keywords:
        kw_lower = str(kw).lower()

        if not kw_lower:
            continue

        idx = text_lower.find(kw_lower)

        if idx != -1:
            start = max(0, idx - window)
            end = min(len(text), idx + len(kw) + window)
            return text[start:end]

    return text[:800]


search_rows = []

for i, row in pages_df.iterrows():
    title = str(row["title"])
    url = str(row["url"])
    text = str(row["text"])

    title_lower = title.lower()
    url_lower = url.lower()
    text_lower = text.lower()

    score = 0
    hit_terms = []

    for kw in search_keywords:
        kw_lower = str(kw).lower()

        if not kw_lower:
            continue

        if kw_lower in title_lower:
            score += 10
            hit_terms.append(kw)

        if kw_lower in url_lower:
            score += 6
            hit_terms.append(kw)

        if kw_lower in text_lower:
            score += 2
            hit_terms.append(kw)

    snippet = make_snippet(text, search_keywords)

    search_rows.append({
        "score": score,
        "hit_terms": ", ".join(sorted(set(hit_terms))),
        "title": title,
        "url": url,
        "snippet": snippet
    })

search_df = pd.DataFrame(search_rows)
search_df = search_df.sort_values(by="score", ascending=False).reset_index(drop=True)

TOP_N = 5
candidate_pages_df = search_df.head(TOP_N).copy()
candidate_pages_df["evidence_no"] = candidate_pages_df.index + 1

print("\n관련 후보 페이지:")
display(candidate_pages_df[["evidence_no", "score", "hit_terms", "title", "url"]])

if candidate_pages_df["score"].max() == 0:
    print("\n주의: 검색 점수가 모두 0입니다. 홈페이지 내부에서 질문 관련 키워드를 찾지 못했을 수 있습니다.")

상위 5개 후보 페이지를 근거로만 답변을 생성

In [ ]:
evidence_text = ""

for _, row in candidate_pages_df.iterrows():
    evidence_text += f"""
[근거 번호: {row["evidence_no"]}]
[검색 점수] {row["score"]}
[매칭어] {row["hit_terms"]}
[페이지 제목] {row["title"]}
[URL] {row["url"]}
[본문 일부] {row["snippet"]}
---
"""

final_prompt = ChatPromptTemplate.from_template("""
너는 홈페이지 내부 검색 기반 QA Agent다.

사용자 질문:
{user_question}

검색 대상 홈페이지:
{homepage_url}

아래는 홈페이지 내부 페이지를 검색해서 얻은 후보 근거다.

[근거 후보]
{evidence_text}

규칙:
- 반드시 근거 후보에 있는 내용만 사용하라.
- 근거에 없으면 "홈페이지 검색 결과만으로는 확인되지 않습니다"라고 답하라.
- 답변에는 근거 번호와 URL을 함께 제시하라.
- 외부 검색, 위키피디아, 일반 지식은 사용하지 말라.
- 한국어로 답하라.

[출력 형식]

# 홈페이지 검색 기반 답변

## 1. 결론
- 질문에 대한 답을 먼저 간단히 말하라.

## 2. 근거
- 근거 번호:
- 페이지 제목:
- URL:
- 확인한 내용:

## 3. ReAct 흐름
- Question:
- Plan:
- Action:
- Observation:
- Final Answer:

## 4. 한계
- 홈페이지 내부 검색 결과만 사용했기 때문에 생기는 한계를 말하라.
""")

final_chain = final_prompt | llm | StrOutputParser()

final_answer = final_chain.invoke({
    "user_question": user_question,
    "homepage_url": homepage_url,
    "evidence_text": evidence_text
})

print("\n" + "=" * 100)
print("[Final Output] Simple Homepage QA ReAct Agent")
print("=" * 100)
print(final_answer)